# Install and Run Perseus MCP

This notebook explains the supported ways to install, launch, configure, update, and remove the Perseus MCP server. It is an onboarding guide rather than a text-research workflow; continue with notebook `01_` or `03_` after choosing an installation method.

## Table of contents

1. Understand the installation choices
2. Install from PyPI with pip
3. Install and run with uv
4. Clone and run the repository locally
5. Choose a launch command
6. Configure an MCP client
7. Verify the installed package
8. Test with MCP Inspector
9. Update or uninstall
10. Troubleshoot common setup problems
11. Continue with the research notebooks

## 1 - Understand the installation choices

Perseus MCP is a **local stdio MCP server**. An MCP-capable application launches the server as a child process and communicates with it over standard input and output. The server contacts the public Perseus CTS and Scaife services; it does not download the full corpus and does not require an API key.

| Method | Best for | What the client launches |
|---|---|---|
| PyPI in a virtual environment | Most users and stable releases | `perseus-mcp` or `python -m perseus_mcp` |
| `uv tool install` | An isolated command available on your PATH | `perseus-mcp` |
| `uvx` | Trying the latest published package without a persistent install | `uvx perseus-mcp` |
| Editable repository install | Contributors changing code or notebooks | The editable `perseus-mcp` command |
| Repository-local `uv run` | Running directly from a clone | `uv --directory ... run perseus-mcp` |

Use only one method at first. Multiple installations can make it unclear which executable an MCP client is launching.

## 2 - Install from PyPI with pip

A virtual environment keeps Perseus MCP and its dependencies separate from other Python projects.

### macOS or Linux

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install perseus-mcp
```

### Windows PowerShell

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install perseus-mcp
```

The installation provides both a `perseus-mcp` console command and the Python module entry point `python -m perseus_mcp`.

## 3 - Install and run with uv

[uv](https://docs.astral.sh/uv/) can install the command into an isolated tool environment:

```bash
uv tool install perseus-mcp
perseus-mcp
```

To try the published package without keeping a tool installation:

```bash
uvx perseus-mcp
```

`uvx` is convenient for testing, but an explicit installed command or absolute interpreter path is more predictable in long-lived desktop-client configuration.

## 4 - Clone and run the repository locally

Use this method when developing the server, running the repository notebooks, or testing unreleased changes.

```bash
git clone https://github.com/tonyjurg/Perseus-mcp.git
cd Perseus-mcp
uv sync
uv run perseus-mcp
```

An editable pip installation is equivalent for development:

```bash
python -m venv .venv
# Activate the environment, then:
python -m pip install -e ".[dev]"
perseus-mcp
```

The implementation lives in `src/perseus_mcp/server.py`. Launch it through the installed `perseus-mcp` command or the `python -m perseus_mcp` module entry point.

## 5 - Choose a launch command

Running the server directly normally appears to wait without printing a prompt. That is expected: stdio MCP servers wait for an MCP client to send protocol messages. Stop a manual run with `Ctrl+C`.

Supported launch forms:

```bash
# Installed console command
perseus-mcp

# Installed package through a particular interpreter
python -m perseus_mcp

# Repository-local uv environment
uv --directory /full/path/to/Perseus-mcp run perseus-mcp

# Published package in an ephemeral uv environment
uvx perseus-mcp
```

For desktop applications, `python -m perseus_mcp` with an **absolute path to the virtual environment's Python executable** is usually the least ambiguous option.

## 6 - Configure an MCP client

Most MCP clients expect a server name, executable, argument list, and optional environment variables.

### Installed package using an absolute Python path

```json
{
  "mcpServers": {
    "perseus": {
      "command": "/absolute/path/to/.venv/bin/python",
      "args": ["-m", "perseus_mcp"],
      "env": {}
    }
  }
}
```

On Windows, `command` will resemble `C:\\full\\path\\to\\.venv\\Scripts\\python.exe`.

### Repository clone using uv

```json
{
  "mcpServers": {
    "perseus": {
      "command": "uv",
      "args": [
        "--directory",
        "/full/path/to/Perseus-mcp",
        "run",
        "perseus-mcp"
      ],
      "env": {}
    }
  }
}
```

Replace all example paths with absolute paths. Restart the MCP client after changing its configuration.

The next cell generates a configuration using the Python interpreter that runs this notebook. It only works as an MCP launch command if `perseus-mcp` is installed in this same environment.

In [ ]:
import json
import sys

mcp_config = {
    "mcpServers": {
        "perseus": {
            "command": sys.executable,
            "args": ["-m", "perseus_mcp"],
            "env": {},
        }
    }
}

print(json.dumps(mcp_config, indent=2))

## 7 - Verify the installed package

Run the next cell in the environment where Perseus MCP is installed. It checks package metadata, import location, and whether the console command is visible on `PATH`. It does not start the server or make network requests.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from shutil import which

try:
    installed_version = version("perseus-mcp")
except PackageNotFoundError:
    installed_version = None

package_spec = find_spec("perseus_mcp")
console_command = which("perseus-mcp")

print(f"Installed distribution version: {installed_version or 'not installed'}")
print(f"Importable package location: {package_spec.origin if package_spec else 'not importable'}")
print(f"Console command: {console_command or 'not found on PATH'}")

assert installed_version is not None, "Install perseus-mcp in this kernel's Python environment."
assert package_spec is not None

## 8 - Test with MCP Inspector

MCP Inspector starts the server, performs the MCP handshake, and provides a browser interface for listing and calling tools.

For an installed command:

```bash
npx @modelcontextprotocol/inspector perseus-mcp
```

For a repository clone:

```bash
npx @modelcontextprotocol/inspector uv --directory /full/path/to/Perseus-mcp run perseus-mcp
```

After connecting, confirm that tools such as `find_author_names`, `get_author_resources`, `get_passage_plaintext`, and `search_perseus` are listed.

## 9 - Update or uninstall

### pip installation

```bash
python -m pip install --upgrade perseus-mcp
python -m pip uninstall perseus-mcp
```

### uv tool installation

```bash
uv tool upgrade perseus-mcp
uv tool uninstall perseus-mcp
```

### Repository clone

```bash
git pull
uv sync
```

Restart the MCP client after upgrading so it launches the new process.

## 10 - Troubleshoot common setup problems

| Symptom | Likely cause and response |
|---|---|
| `perseus-mcp` is not found | The environment is not activated or its scripts directory is not on `PATH`; use the absolute Python path with `-m perseus_mcp` |
| `No module named perseus_mcp` | The client is launching a different Python interpreter; install into that interpreter or correct `command` |
| The command appears to hang | Normal for a stdio server waiting for MCP messages; test with MCP Inspector instead |
| The client shows no tools | Restart it, validate JSON syntax, use absolute paths, and run the same command manually to expose startup errors |
| A local code edit is ignored | The client is launching the PyPI installation instead of the editable checkout, or a stale process is still running |
| Cache files appear in an unexpected directory | Set `PERSEUS_MCP_CACHE_DIR` to an absolute writable location in the client `env` object |
| Passage or search calls fail after connection | The local server is running, but the public Perseus or Scaife service may be unavailable or the URN/query may be invalid |

The Perseus MCP server itself needs no OpenRouter, Anthropic, OpenAI, or other model-provider key. Model credentials belong to the MCP host or to optional LLM demonstration notebooks.

## 11 - Continue with the research notebooks

- [`01_basic_cts_workflow.ipynb`](01_basic_cts_workflow.ipynb) explains the underlying CTS HTTP service.
- [`02_search_and_navigation.ipynb`](02_search_and_navigation.ipynb) explains direct Perseus and Scaife search concepts.
- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) makes the first in-process MCP tool calls.
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) catalogs the complete live MCP tool surface.

Project resources:

- [PyPI package](https://pypi.org/project/perseus-mcp/)
- [GitHub repository](https://github.com/tonyjurg/Perseus-mcp)
- [End-user guide](https://tonyjurg.github.io/Perseus-mcp/enduser/)
- [MCP Inspector](https://github.com/modelcontextprotocol/inspector)

## Notebook version

| Field | Value |
|---|---|
| Author | Tony Jurg |
| Version | 1.0 |
| Date | June 18, 2026 |